Understanding PyTorch Buffers
In essence, PyTorch buffers are tensor attributes associated with a PyTorch module or model similar to parameters, but unlike parameters, buffers are not updated during training.

Buffers in PyTorch are particularly useful when dealing with GPU computations, as they need to be transferred between devices (like from CPU to GPU) alongside the model's parameters. Unlike parameters, buffers do not require gradient computation, but they still need to be on the correct device to ensure that all computations are performed correctly.

In chapter 3, we use PyTorch buffers via self.register_buffer, which is only briefly explained in the book. Since the concept and purpose are not immediately clear, this code notebook offers a longer explanation with a hands-on example.

An example without buffers
Suppose we have the following code, which is based on code from chapter 3. This version has been modified to exclude buffers. It implements the causal self-attention mechanism used in LLMs:

In [ ]:
import torch
import torch.nn as nn

# 这是一个“不使用 buffer”的版本，用来对比说明 register_buffer 的作用。
# 关键点：self.mask 在这里只是一个普通的 tensor 属性（既不是 nn.Parameter，
# 也没有通过 self.register_buffer 注册），因此它不会被 PyTorch 的 nn.Module
# 机制“看见”，不会出现在 state_dict 里，也不会随 model.to(device) 一起搬到 GPU 上。
class CausalAttentionWithoutBuffers(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        # W_query / W_key / W_value 是 nn.Linear，其内部权重是 nn.Parameter，
        # 会自动注册到模块中，参与梯度更新，也会随 .to(device) 自动搬运。
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # 因果掩码（causal mask）：上三角为 1（不含对角线），用于屏蔽“未来”的 token。
        # 注意：这里直接赋值给 self.mask，只是给模块加了一个普通属性，
        # PyTorch 并不知道它的存在（不参与梯度，也不会随模型迁移设备）。
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 计算注意力分数（Q @ K^T）
        attn_scores = queries @ keys.transpose(1, 2)
        # 用因果掩码把“未来位置”的注意力分数填成 -inf，softmax 后其权重趋近于 0，
        # 从而保证当前位置只能看到自己及之前的 token（因果性/自回归特性）。
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)

# 6 个 token，每个 token 用 3 维向量表示（模拟词嵌入）
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# 把同一个序列复制成 batch_size=2 的 batch，形状为 [2, 6, 3]
batch = torch.stack((inputs, inputs), dim=0)
context_length = batch.shape[1]
d_in = inputs.shape[1]
d_out = 2

# 实例化不含 buffer 的因果自注意力模块
ca_without_buffer = CausalAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)

# 推理阶段关闭梯度计算，节省显存/内存并加速
with torch.no_grad():
    context_vecs = ca_without_buffer(batch)

print(context_vecs)

In [ ]:
has_cuda = torch.cuda.is_available()
has_mps = torch.backends.mps.is_available()

print("Machine has GPU:", has_cuda or has_mps)

if has_mps:
    device = torch.device("mps")   # Apple Silicon GPU (Metal)
elif has_cuda:
    device = torch.device("cuda")  # NVIDIA GPU
else:
    device = torch.device("cpu")   # CPU fallback

print(f"Using device: {device}")

# 把输入数据搬到目标设备
batch = batch.to(device)
# .to(device) 只会搬运模块中被 PyTorch 追踪到的张量：
# 即 nn.Parameter（如 W_query/W_key/W_value 的权重）以及通过 register_buffer
# 注册过的 buffer。而 self.mask 只是普通属性，不会被这次调用搬到 device 上，
# 它仍然留在 CPU（这正是本 notebook 要演示的“陷阱”）。
# 注意：如果当前机器既没有 GPU 也没有 MPS（device 仍为 cpu），下面几个单元
# 演示的“设备不一致”问题就不会触发，因为 mask 本来就在 CPU 上。
ca_without_buffer = ca_without_buffer.to(device)

In [ ]:
# 在有 GPU/MPS 的机器上，这里通常会因为 queries/keys 等张量已在 device 上，
# 而 self.mask 仍停留在 CPU，导致 masked_fill_ 等操作出现设备不匹配的报错
# （RuntimeError: Expected all tensors to be on the same device）。
# 在纯 CPU 环境下运行本 notebook 则不会复现这个问题，因为 mask 本来就在 CPU。
with torch.no_grad():
    context_vecs = ca_without_buffer(batch)

print(context_vecs)

In [ ]:

# W_query.weight 是 nn.Parameter，会随 .to(device) 自动搬运到目标设备
print("W_query.device:", ca_without_buffer.W_query.weight.device)
# mask 只是普通 tensor 属性，不受 .to(device) 影响，通常仍停留在原设备（CPU）上，
# 与上面的权重设备不一致
print("mask.device:", ca_without_buffer.mask.device)

In [ ]:
type(ca_without_buffer.mask)  # 输出 torch.Tensor：说明它只是普通张量属性，未被 nn.Module 追踪

In [ ]:
# 由于 mask 不是 buffer，无法通过 model.to(device) 自动跟随迁移，
# 只能像这样手动把它搬到目标设备，才能让后续计算的张量设备保持一致。
# 如果模型中有多个这样的普通张量属性，就需要逐个手动处理，容易遗漏出错——
# 这正是应该改用 register_buffer 的原因。
ca_without_buffer.mask = ca_without_buffer.mask.to(device)
print("mask.device:", ca_without_buffer.mask.device)

In [ ]:
# 经过上面手动搬运 mask 之后，所有张量设备一致，前向传播可以正常执行
with torch.no_grad():
    context_vecs = ca_without_buffer(batch)

print(context_vecs)

An example with buffers

In [ ]:
import torch
import torch.nn as nn

# 这是使用 register_buffer 的版本。相比上面的版本，唯一的区别就是
# mask 的注册方式：不再用 self.mask = ... 直接赋值，而是通过
# self.register_buffer("mask", ...) 把它注册为模块的 buffer。
class CausalAttentionWithBuffer(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # Old:
        # self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

        # New:
        # register_buffer(name, tensor) 会把这个张量注册为模块的“buffer”：
        # - 和 nn.Parameter 类似，会出现在 model.state_dict() 中，会随
        #   model.to(device) / model.cuda() 等调用一起自动搬运设备；
        # - 但和 nn.Parameter 不同，buffer 默认 requires_grad=False，
        #   不会被优化器更新，也不会参与反向传播的梯度计算。
        # 因果掩码正是典型的“需要跟着模型走、但不需要训练”的张量，非常适合用 buffer。
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
# 因为 mask 是通过 register_buffer 注册的 buffer，调用 .to(device) 时会和
# nn.Parameter 一样自动被搬运到目标设备，不需要像前面那样手动搬运
ca_with_buffer.to(device)

print("W_query.device:", ca_with_buffer.W_query.weight.device)
# 可以看到 mask 与权重的设备保持一致，说明 buffer 成功随模型一起迁移
print("mask.device:", ca_with_buffer.mask.device)

In [ ]:
with torch.no_grad():
    context_vecs = ca_with_buffer(batch)

print(context_vecs)

Buffers and state_dict

In [ ]:
ca_without_buffer.state_dict()  # mask 只是普通属性，不会出现在 state_dict 里（也就不会被保存/加载）

In [ ]:
ca_with_buffer.state_dict()  # mask 作为 buffer 被注册，会出现在 state_dict 中，可随模型一起保存/加载

In [ ]:
# buffer 本质上还是一个普通的 tensor，可以像普通张量一样原地修改其中的值
# （这不会影响它“是 buffer”这一身份，只是改变了它当前存储的数值）
ca_with_buffer.mask[ca_with_buffer.mask == 1.] = 2.
ca_with_buffer.mask

In [ ]:
# 因为 mask 是 state_dict 的一部分，torch.save 会把上面修改后的值（2.）也保存下来
torch.save(ca_with_buffer.state_dict(), "model.pth")

# 新建一个模型实例：此时它的 mask 是刚初始化的默认值（上三角为 1）
new_ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
# 加载 state_dict 后，mask 会被覆盖为之前保存的值（包含我们手动改过的 2.）
new_ca_with_buffer.load_state_dict(torch.load("model.pth"))

# 可以看到加载后的 mask 保留了修改后的数值，证明 buffer 会被 state_dict 保存/恢复
new_ca_with_buffer.mask


In [ ]:
ca_without_buffer.mask[ca_without_buffer.mask == 1.] = 2.

# 因为 mask 不是 buffer，不会被包含在 state_dict 中，所以这里保存的 state_dict
# 里根本没有 mask 这一项
torch.save(ca_without_buffer.state_dict(), "model.pth")

# 新建的模型实例的 mask 是重新初始化的默认值（上三角为 1）
new_ca_without_buffer = CausalAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)
# load_state_dict 不会（也无法）恢复 mask，因为它从未被保存过
new_ca_without_buffer.load_state_dict(torch.load("model.pth"))

# 因此这里打印出来的仍是新实例初始化时的默认 mask（值为 1），
# 而不是我们之前手动修改过的值（2.）——这正是不用 buffer 的代价
new_ca_without_buffer.mask